# Evaluate RAG on a Real HR Policy PDF (Free, Ollama only)

**Series:** Chunking -> Embeddings -> Vector Database -> RAG -> **Evaluation on a real PDF (today)**


**The PDF:** *Human Resources Policy Manual 2016* by Encina Wastewater Authority (a public agency in California, USA). 
It is freely available on the agency's website. It is a real manual, with real rules about working hours, breaks, overtime, probation, holidays, passwords and more. 
(It is a US document, but all the steps are the same for **your own company's HR policy PDF**.)

## What we will do

1. Download the PDF and read the text
2. Cut the text into chunks and store the chunks in ChromaDB
3. Make a **test set** of questions with known answers
4. Measure retrieval (**Hit Rate** and **MRR**)
5. Compare different chunk sizes and Top-K values
6. Use **Ollama** to answer the questions and to **judge** the answers

**Before you start:** Ollama must be running, and you need the model once: `ollama pull llama3.2`

In [2]:
!pip install -q pypdf requests sentence-transformers chromadb ollama

In [16]:
import os
import requests
import chromadb
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import textwrap

model = SentenceTransformer("all-MiniLM-L6-v2")
db = chromadb.Client()            # in-memory database
print("Ready!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6201.12it/s]


Ready!


## Step 1: Download the PDF

In [17]:
PDF_URL  = "https://www.encinajpa.com/images/pdf/2016_HR_Policy_Manual.pdf"
PDF_FILE = "hr_policy_manual.pdf"

if not os.path.exists(PDF_FILE):
    response = requests.get(PDF_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
    response.raise_for_status()
    with open(PDF_FILE, "wb") as f:
        f.write(response.content)

print("PDF saved:", PDF_FILE, "|", os.path.getsize(PDF_FILE) // 1024, "KB")

PDF saved: hr_policy_manual.pdf | 2356 KB


> **If the download fails** (for example, a firewall): open the `PDF_URL` in your browser, save the file as `hr_policy_manual.pdf` in the same folder as this notebook, and run the cell again.

## Step 2: Read the text from the PDF

In [18]:
reader = PdfReader(PDF_FILE)

text = ""
for page in reader.pages:
    text = text + " " + (page.extract_text() or "")

words = text.split()          # the whole manual as a list of words

print("Pages:", len(reader.pages))
print("Words:", len(words))
print()
print("First 60 words:")
print(" ".join(words[:60]))

Pages: 102
Words: 36749

First 60 words:
Human Resources Policy Manual 2016 Encina Wastewater Authority Table of Contents Page Section 1 Introduction and General Provisions 1.1 Purpose of EWA Human Resources Policy Manual (HRPM) 1 1.2 Scope of HRPM 1 1.3 Administration 1 1.4 Distribution of the HRPM 1 Section 2 Definition of Terms 2.1 Definition of an Employee 2 2.2 Definition of Nonexempt and Exempt Employee


## Step 3: Chunking and the vector database

A real PDF has no neat list of sentences. So we use the **simplest** chunking: 

> Take **N words** at a time. Move forward, but keep a small **overlap** (20%), so a fact is not cut in half.

| Function | Job |
|---|---|
| `make_chunks` | Cut the words into chunks of `size` words |
| `build_collection` | Make chunks, convert them to vectors, store in ChromaDB |
| `retrieve` | Find the Top-K chunks for a question |

In [19]:
def make_chunks(words, size=100):
    overlap = size // 5                       # 20% overlap, so a fact is not cut in half
    step = size - overlap
    chunks = []
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start:start + size]))
    return chunks


def build_collection(size=100):
    all_chunks = make_chunks(words, size)
    vectors = model.encode(all_chunks, normalize_embeddings=True).tolist()
    ids = [str(i) for i in range(len(all_chunks))]

    col = db.get_or_create_collection(f"hr_manual_{size}")
    col.upsert(ids=ids, documents=all_chunks, embeddings=vectors)
    return col


def retrieve(col, question, k=3):
    q_vec = model.encode([question], normalize_embeddings=True).tolist()
    result = col.query(query_embeddings=q_vec, n_results=k)
    return result["documents"][0]             # best chunk first


col = build_collection(size=100)
print("Chunks stored:", col.count())
print()
print("Example chunk:")
chunk = retrieve(col, "How long is the probation period?", k=1)[0]
print(textwrap.fill(chunk, width=90))         # show the chunk in many short lines

Chunks stored: 460

Example chunk:
the probationary period without cause and without the right of appeal. The department
director must first notify the General Manager in writing of the recommendation to remove
an employee during probation. Upon app roval by the General Manager, the department
director shall provide prompt notification to the appointee. 6.3 Promotion During
Probationary Period An appointee may be promoted to a new position in another
classification during a probationary period. If an appointee is promoted during this
period, the probationary period shall begin anew, beginning with the date of appointment
to the new position. 6.4 Completion of Probationary Period Prior to


## Step 4: Make the test set

For each question we write:

- `question`: written in our own words
- `key_fact`: a short piece of text that **must** be inside the retrieved chunk (we copied it from the PDF)
- `answer`: the correct answer, in simple words (the judge will use it later)

We also add **2 questions that are NOT in the manual**. A good system must say "I don't know".

In [24]:
test_set = [
    {"question": "How many hours per week must someone be scheduled to work to be a regular full-time employee?",
     "key_fact": "at least 30 hours per week", "answer": "At least 30 hours per week."},
    {"question": "What is the maximum number of hours an intern can work in a fiscal year?",
     "key_fact": "1000 hours per fiscal year", "answer": "1000 hours per fiscal year."},
    {"question": "How long is the probation period for a newly hired employee?",
     "key_fact": "12 months from the date of employment", "answer": "12 months."},
    {"question": "How far from the building entrance must a smoking area be?",
     "key_fact": "20 feet away", "answer": "At least 20 feet away from a main entrance, exit and windows."},
    {"question": "How soon must an employee report a criminal drug conviction?",
     "key_fact": "within five days after the conviction", "answer": "Within five days after the conviction."},
    {"question": "How often must user-level passwords be changed?",
     "key_fact": "every four months", "answer": "At least every four months."},
    {"question": "Who chooses the doctor for a work injury in the first month?",
     "key_fact": "first 30 days after an injury", "answer": "The employer (EWA) decides which medical provider treats the employee."},
    {"question": "What is the minimum age to be hired?",
     "key_fact": "18 years of age", "answer": "18 years of age."},
    {"question": "How soon must the I-9 form be completed after a person is hired?",
     "key_fact": "within three days of the hire date", "answer": "Within three days of the hire date."},
    {"question": "How long can a temporary assignment to a higher job role last?",
     "key_fact": "may not exceed a period of six months", "answer": "Not more than six months, unless the General Manager approves."},
    {"question": "What is the usual minimum pay raise when someone is promoted?",
     "key_fact": "minimum of five percent", "answer": "Typically a minimum of five percent."},
    {"question": "How much notice is needed before an employee's work shift is changed?",
     "key_fact": "48 hours prior written notice", "answer": "48 hours of prior written notice."},
    {"question": "How early must I tell my supervisor if I will be late or absent?",
     "key_fact": "30 minutes before the beginning of their workday", "answer": "No later than 30 minutes before the workday or shift starts."},
    {"question": "When does an employee get an unpaid meal break?",
     "key_fact": "more than five hours", "answer": "A 30-minute unpaid meal break when working more than five hours a day."},
    {"question": "How long are the rest breaks for non-exempt employees?",
     "key_fact": "rest period for every four hours", "answer": "A ten-minute rest period for every four hours worked."},
    {"question": "After how many hours in a week does overtime pay start?",
     "key_fact": "in excess of 40 hours", "answer": "After 40 hours in the workweek."},
    {"question": "What is the maximum meal reimbursement when someone works extra hours?",
     "key_fact": "not to exceed $15", "answer": "Not more than $15."},
    {"question": "How much is the pay for working on Christmas Day?",
     "key_fact": "two times their regular rate of pay", "answer": "Two times the regular rate of pay for each hour worked, plus holiday pay."},
    {"question": "How often are employees paid?",
     "key_fact": "every other Wednesday", "answer": "Every two weeks, on Wednesday."},
    {"question": "How much notice is required to resign in good standing?",
     "key_fact": "not less than two weeks prior", "answer": "At least two weeks before the last day."},
    # These 2 are NOT in the manual -> the system should say "I don't know"
    {"question": "How many stock options do new employees receive?", "key_fact": None, "answer": None},
    {"question": "What is the monthly gym membership allowance?",     "key_fact": None, "answer": None},
]
answerable = [t for t in test_set if t["key_fact"]]
print("Total questions:", len(test_set), "| Answerable:", len(answerable))

Total questions: 22 | Answerable: 20


### Check the test set (very important!)

Before we measure anything, we must check that every `key_fact` really exists in the PDF text. 
Otherwise a question can never succeed, and our score will be unfair.

We write a small helper `clean()` that makes text lowercase and **removes all spaces and line breaks**. 
This way, PDF line breaks do not break our matching.

In [25]:
def clean(text):
    return "".join(text.lower().split())      # lowercase, remove all spaces and line breaks


whole_pdf = clean(" ".join(words))
problems = 0
for t in answerable:
    if clean(t["key_fact"]) not in whole_pdf:
        print("NOT FOUND IN PDF TEXT:", t["key_fact"], "|", t["question"])
        problems += 1

print("Key facts missing from the PDF:", problems)

Key facts missing from the PDF: 0


> If some key facts are missing, the PDF text was read slightly differently on your computer. 
> Open the PDF, find the sentence, and fix the `key_fact` in the test set (keep it short). Then run the check again.

## Step 5: Evaluate retrieval

For each question we get the Top-K chunks and look for the `key_fact` inside them.

**Rank** = position of the first chunk that has the key fact (1 = best).

| Metric | Meaning | Example |
|---|---|---|
| **Hit Rate** | In how many questions was the right chunk **anywhere** in the Top-K? | 15 of 20 = 0.75 |
| **MRR** | How **high** was the right chunk? (rank 1 = 1.0, rank 2 = 0.5, rank 3 = 0.33, not found = 0) | average of these scores |

In [26]:
def find_rank(retrieved_chunks, key_fact):
    for position, chunk in enumerate(retrieved_chunks, start=1):
        if clean(key_fact) in clean(chunk):
            return position
    return None                                    # not found


def evaluate_retrieval(col, k=3, show=False):
    hits, scores = 0, []
    for t in answerable:
        rank = find_rank(retrieve(col, t["question"], k), t["key_fact"])
        if rank:
            hits += 1
            scores.append(1 / rank)
        else:
            scores.append(0)
        if show:
            print(f"{'OK  ' if rank else 'FAIL'} rank={rank} | {t['question']}")
    return hits / len(answerable), sum(scores) / len(scores)


hit_rate, mrr = evaluate_retrieval(col, k=3, show=True)
print(f"\nHit Rate = {hit_rate:.2f}   MRR = {mrr:.2f}")

OK   rank=3 | How many hours per week must someone be scheduled to work to be a regular full-time employee?
OK   rank=1 | What is the maximum number of hours an intern can work in a fiscal year?
OK   rank=3 | How long is the probation period for a newly hired employee?
OK   rank=1 | How far from the building entrance must a smoking area be?
OK   rank=1 | How soon must an employee report a criminal drug conviction?
OK   rank=1 | How often must user-level passwords be changed?
OK   rank=2 | Who chooses the doctor for a work injury in the first month?
OK   rank=1 | What is the minimum age to be hired?
OK   rank=1 | How soon must the I-9 form be completed after a person is hired?
OK   rank=2 | How long can a temporary assignment to a higher job role last?
OK   rank=1 | What is the usual minimum pay raise when someone is promoted?
OK   rank=1 | How much notice is needed before an employee's work shift is changed?
FAIL rank=None | How early must I tell my supervisor if I will be late or abse

## Step 6: Experiments (change ONE thing at a time)

We compare:
- **chunk size** = 50, 100, 200 words
- **Top-K** = 1, 3, 5

This cell can take a minute or two, because it builds 3 databases from the whole PDF.

In [27]:
print(f"{'chunk size':<12}{'chunks':<9}{'k':<4}{'Hit Rate':<11}{'MRR':<8}")
print("-" * 44)
for size in [50, 100, 200]:
    col_s = build_collection(size=size)
    for k in [1, 3, 5]:
        h, m = evaluate_retrieval(col_s, k=k)
        print(f"{size:<12}{col_s.count():<9}{k:<4}{h:<11.2f}{m:<8.2f}")

chunk size  chunks   k   Hit Rate   MRR     
--------------------------------------------
50          919      1   0.45       0.45    
50          919      3   0.80       0.60    
50          919      5   0.90       0.62    
100         460      1   0.45       0.45    
100         460      3   0.80       0.60    
100         460      5   0.85       0.61    
200         230      1   0.55       0.55    
200         230      3   0.75       0.64    
200         230      5   0.90       0.67    


### How to read the table

- **Higher K** usually gives a higher Hit Rate, but sends **more text** to the LLM (slower, more noise).
- **Bigger chunks** can raise the Hit Rate, but a chunk may contain **other topics** and confuse the LLM.
- **Very small chunks** may not have enough context.
- Do not only pick the highest number. Pick a **good score with a small K and a small chunk size**, and also check the answers (Step 8).

## Step 7: Look at the failures

The numbers tell us **how many** questions fail. Now we see **which** ones and **why**.

In [28]:
def show_failures(col, k=3):
    for t in answerable:
        retrieved = retrieve(col, t["question"], k)
        if find_rank(retrieved, t["key_fact"]) is None:
            print("QUESTION :", t["question"])
            print("NEEDED   :", t["key_fact"])
            print("TOP CHUNK:", retrieved[0][:150], "...")
            print()

show_failures(build_collection(size=100), k=3)

QUESTION : How early must I tell my supervisor if I will be late or absent?
NEEDED   : 30 minutes before the beginning of their workday
TOP CHUNK: 48 hours prior written notice provided to the employee. An exception to the 48 -hour prior written notice is when EWA holds Board of Directors meeting ...

QUESTION : When does an employee get an unpaid meal break?
NEEDED   : more than five hours
TOP CHUNK: in the middle of the four-hour work period prior to lunch, and the second rest period should be taken roughly in the middle of the four -hour work per ...

QUESTION : How long are the rest breaks for non-exempt employees?
NEEDED   : rest period for every four hours
TOP CHUNK: in the middle of the four-hour work period prior to lunch, and the second rest period should be taken roughly in the middle of the four -hour work per ...

QUESTION : After how many hours in a week does overtime pay start?
NEEDED   : in excess of 40 hours
TOP CHUNK: double-time pay for hours worked in excess of 12 in

Common reasons in a real PDF:
- The question uses **different words** than the manual (embedding model did not connect them).
- The **table of contents** or a similar section matched better than the real answer.
- The right chunk is found, but **ranked low** (try a bigger K, or a reranker in a future video).
- The fact sits in a **table** that the PDF reader turned into messy text.

## Step 8: Evaluate the answers (Ollama as the judge)

Good retrieval is only half of the story. Is the **final answer** correct? We use Ollama two ways:

1. **Generate** the answer (RAG).
2. **Judge** the answer with YES or NO.

| Check | Question |
|---|---|
| **Correct?** | Does the answer match the expected answer? |
| **Grounded?** | Does the answer use only the retrieved chunks (no made-up facts)? |
| **Says "I don't know"?** | For the 2 questions that are not in the manual |

> A small local model is not a perfect judge. Read a few results yourself.

In [29]:
import ollama

LLM = "llama3.2"

def ask_llm(prompt):
    response = ollama.chat(model=LLM, messages=[{"role": "user", "content": prompt}],
                           options={"temperature": 0})
    return response["message"]["content"].strip()


def rag_answer(col, question, k=3):
    chunks = retrieve(col, question, k)
    context = "\n".join(f"- {c}" for c in chunks)
    prompt = f'''Answer the question using only the context below.
If the answer is not in the context, say: "I don't know based on the documents."

Context:
{context}

Question: {question}'''
    return ask_llm(prompt), context


def judge(prompt):
    return ask_llm(prompt).upper().startswith("YES")


def is_correct(question, expected, answer):
    return judge(f'''Question: {question}
Expected answer: {expected}
Student answer: {answer}

Does the student answer give the same facts as the expected answer?
Reply with only YES or NO.''')


def is_grounded(context, answer):
    return judge(f'''Context:
{context}

Answer: {answer}

Is every fact in the answer supported by the context above? Do not allow new facts.
Reply with only YES or NO.''')

In [30]:
col = build_collection(size=100)
correct, grounded, dont_know_ok = 0, 0, 0

for t in test_set:
    answer, context = rag_answer(col, t["question"], k=3)

    if t["key_fact"] is None:                         # a question that is NOT in the manual
        said_idk = "don't know" in answer.lower() or "do not know" in answer.lower()
        dont_know_ok += said_idk
        print(f"{'OK  ' if said_idk else 'FAIL'} (should say I don't know) | {t['question']}")
        print("     Answer:", answer)
        continue

    c = is_correct(t["question"], t["answer"], answer)
    g = is_grounded(context, answer)
    correct += c
    grounded += g
    print(f"{'OK  ' if c else 'FAIL'} correct={c!s:<5} grounded={g!s:<5} | {t['question']}")
    print("     Answer:", answer)

print()
print(f"Correct answers  : {correct}/{len(answerable)}")
print(f"Grounded answers : {grounded}/{len(answerable)}")
print(f"Said 'I don't know' when needed: {dont_know_ok}/2")

OK   correct=True  grounded=True  | How many hours per week must someone be scheduled to work to be a regular full-time employee?
     Answer: According to the context, a Regular Full-Time Employee is defined as an employee who is hired to fill a full-time position and is regularly scheduled to work at least 30 hours per week.
FAIL correct=False grounded=True  | What is the maximum number of hours an intern can work in a fiscal year?
     Answer: According to the context, interns (temporary employees) are not eligible for any employee benefits with the exception of deferred compensation benefits and California-mandated sick leave. They may not work more than 1000 hours per fiscal year.
FAIL correct=False grounded=True  | How long is the probation period for a newly hired employee?
     Answer: According to Section 6.1 of the context, an employee's probationary period is a minimum of one year, which serves as part of the selection process and is utilized to determine an employee's abili